# 인수 분해 형태와 스킵-트라이그램 버그 - 인수 분해의 한계와 버그

- Tutorial ID: `adv-7-1`
- Tutorial: 인수 분해 형태와 스킵-트라이그램 버그
- Section ID: `adv-7-1-1`
- Section: 인수 분해의 한계와 버그


In [ ]:
# ============================================================
# 코드 읽는 법 — 인수 분해의 한계와 버그
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 60)
print("인수 분해 형태와 스킵-트라이그램 버그")
print("=" * 60)

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()

vocab = ['keep', 'in', 'at', 'mind', 'bay', 'on', 'hand']
V = len(vocab)

np.random.seed(42)
C_QK = np.random.randn(V, V) * 0.5
C_OV = np.random.randn(V, V) * 0.5

keep, in_i, at, mind, bay = [vocab.index(w) for w in ['keep','in','at','mind','bay']]

# 패턴 학습
C_QK[in_i, keep] = 3.0   # 'in'이 'keep' 주목
C_QK[at, keep] = 3.0     # 'at'이 'keep' 주목
C_OV[keep, mind] = 3.0   # 'keep'이 'mind' 예측
C_OV[keep, bay] = 3.0    # 'keep'이 'bay' 예측

# 스킵-트라이그램 점수
def skip_score(query, source, next_tok):
    return C_QK[query, source] * C_OV[source, next_tok]

print("\n의도한 패턴:")
print(f"  'keep...in→mind': {skip_score(in_i, keep, mind):.1f} ✓")
print(f"  'keep...at→bay':  {skip_score(at, keep, bay):.1f} ✓")

print("\n의도치 않은 교차 (버그!):")
print(f"  'keep...in→bay':  {skip_score(in_i, keep, bay):.1f} ✗")
print(f"  'keep...at→mind': {skip_score(at, keep, mind):.1f} ✗")

print("""
원인: 인수 분해 구조
  score = QK[query, source] × OV[source, next]
  
  두 패턴이 같은 source('keep') 공유
  → QK와 OV가 독립적이므로 모든 조합이 활성화!
  → 이것이 1-레이어 모델의 구조적 한계
  
  논문: "이러한 버그는 사소해 보이지만,
  해석성으로 모델 실패를 이해하는 초기 사례"
""")

# 실제 예측 확률
print("'keep ... in ?' 예측:")
sources = [keep]
attn = softmax(np.array([C_QK[in_i, s] for s in sources]))
logits = np.zeros(V)
for i, s in enumerate(sources):
    logits += attn[i] * C_OV[s]
probs = softmax(logits)
for i, (v, p) in enumerate(zip(vocab, probs)):
    if p > 0.05:
        tag = "✓" if v == 'mind' else "✗ (버그)"
        print(f"  {v:8s}: {p:.3f} {tag}")